In [1]:
# 0. Diagnostic / Test Cell
# Ensure that requests and bs4 are functioning correctly before full logic execution
import requests
from bs4 import BeautifulSoup
import re

test_url = "https://www.consumercomplaints.in/?search=UPI+fraud&page=1"
test_headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36"}

print("Fetching diagnostic page...")
response = requests.get(test_url, headers=test_headers)
if response.status_code == 200:
    print("Success! Parsing HTML...")
    soup = BeautifulSoup(response.text, 'html.parser')
    complaints = soup.find_all('div', class_='complaint-box-results')
    print(f"Found {len(complaints)} complaints on page 1.\n")
    if complaints:
        print("--- HTML Snippet of first complaint ---\n")
        print(complaints[0].prettify()[:1000])
        print("\n--- Parsed Entities ---")
        title_tag = complaints[0].find('a', class_='complaint-box-results__title')
        print(f"Title: {title_tag.get_text(separator=' ', strip=True) if title_tag else 'None'}")
        print(f"Link: {title_tag['href'] if title_tag and title_tag.has_attr('href') else 'None'}")
else:
    print(f"Failed diagnostic request with status code: {response.status_code}")

Fetching diagnostic page...
Success! Parsing HTML...
Found 25 complaints on page 1.

--- HTML Snippet of first complaint ---

<div class="white-box complaint-box-results" id="s3535409">
 <div class="complaint-box-results__header">
  <a class="complaint-box-results__title" href="/telegram-upi-fraud-scam-im-complaining-about-earnings-and-withdrawal-system-on-yr67-in-c3535409">
   <strong class="complaint-box-results__title-strong">
    Telegram /
    <span style="background-color:yellow">
     UPI Fraud
    </span>
    / Scam
   </strong>
   — I'm complaining about: Earnings and withdrawal system on yr67.in
  </a>
  <span class="complaint-box-results__text">
   (complaint)
  </span>
 </div>
 <div class="author-box" onclick="document.location.href='/profile-4703493';return false;">
  <div class="author-box__avatar">
   <div class="author-box__avatar A">
    A
   </div>
  </div>
  <div class="author-box__column">
   <div class="author-box__row-profile">
    <div class="author-box__user">
 

# ConsumerComplaints.in Scraper

This notebook scrapes fraud-related complaints from ConsumerComplaints.in based on specific keywords.
It uses `requests` and `BeautifulSoup` as the site uses static HTML and block-level tags.

In [11]:
!pip install selenium pandas beautifulsoup4

In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
import time
import random
import os
from datetime import datetime

# 1. Configuration & Setup
KEYWORD_TAXONOMY = {
    "General Cybercrime": ["cyber crime", "cybercrime", "cyber fraud", "online fraud", "internet fraud", "digital fraud"],
    "Cyber Scam": ["cyber scam", "online scam", "internet scam", "online cheating", "online loot"],
    "Financial Cyber Fraud": ["financial fraud online", "net banking fraud", "e-banking fraud"],
    "UPI Fraud": ["UPI fraud", "UPI scam", "Google Pay fraud", "PhonePe fraud", "Paytm fraud", "BHIM fraud"],
    "QR Code Fraud": ["QR code scam", "QR code fraud", "scan and pay fraud", "fake QR code"],
    "Payment Link Fraud": ["payment link fraud", "collect request scam", "UPI collect fraud", "fake payment link"],
    "Mobile Wallet Fraud": ["mobile wallet fraud", "e-wallet scam", "digital wallet fraud"],
    "OTP Fraud": ["OTP fraud", "OTP scam", "OTP theft", "OTP sharing scam"],
    "SIM Swap": ["SIM swap fraud", "SIM cloning", "SIM swap scam", "duplicate SIM fraud"],
    "KYC Fraud": ["KYC fraud", "KYC scam", "fake KYC", "Aadhaar KYC scam"],
    "Digital Arrest": ["digital arrest", "digital arrest scam", "fake arrest", "video call arrest"],
    "Impersonation Scam": ["police impersonation scam", "CBI fraud call", "customs fraud call", "TRAI scam call", "ED scam call", "fake police call"],
    "Video Call Coercion": ["video call scam", "video call blackmail", "video call extortion", "Skype arrest", "fake interrogation"],
    "Phishing": ["phishing", "phishing attack", "phishing email", "fake website", "spoof website"],
    "Vishing": ["vishing", "voice phishing", "phone scam", "fraud call", "fake bank call"],
    "Smishing": ["smishing", "SMS fraud", "SMS scam", "fake SMS", "phishing SMS"],
    "Loan App Fraud": ["loan app fraud", "instant loan scam", "loan app harassment", "illegal loan app", "Chinese loan app"],
    "Loan App Extortion": ["loan app blackmail", "loan app threat", "morphed photos loan", "contact list harassment", "recovery agent threat"],
    "Investment Scam": ["investment scam", "Ponzi scheme", "online investment fraud", "crypto scam", "bitcoin fraud", "forex trading scam", "pig butchering"],
    "Stock Market Fraud": ["stock market scam", "share trading fraud", "demat fraud", "pump and dump", "trading app scam"],
    "Task Scam": ["task fraud", "part time job scam", "work from home scam", "Telegram task scam", "like and earn", "review task fraud"],
    "Identity Theft": ["identity theft", "identity fraud", "impersonation fraud", "Aadhaar misuse", "PAN fraud"],
    "Data Breach": ["data breach", "data leak", "data theft", "personal data leak", "customer data breach"],
    "Social Engineering": ["social engineering attack", "social engineering fraud", "manipulation scam", "trust scam"],
    "Romance Scam": ["romance scam", "dating fraud", "matrimonial fraud", "love scam", "honey trap", "catfishing fraud"],
    "Sextortion": ["sextortion", "sexual blackmail online", "webcam blackmail", "nude video blackmail"],
    "E-Commerce Fraud": ["e-commerce fraud", "online shopping fraud", "fake product scam", "Flipkart fraud", "Amazon fraud", "refund scam"],
    "Delivery Fraud": ["fake delivery", "courier fraud", "customs duty scam", "parcel scam", "FedEx scam", "delivery OTP scam"],
    "Ransomware": ["ransomware attack", "cyber ransom", "data encryption attack", "file lock malware"],
    "Banking Malware": ["banking trojan", "banking malware", "mobile banking virus", "keylogger fraud", "AnyDesk fraud", "TeamViewer scam", "screen sharing scam"],
    "Deepfake Fraud": ["deepfake scam", "deepfake fraud", "AI voice scam", "voice cloning scam", "fake video call"],
    "Utility Scam": ["electricity bill scam", "utility fraud", "disconnection scam", "fake bill", "power cut threat"],
    "Aadhaar Fraud": ["Aadhaar fraud", "Aadhaar scam", "biometric fraud", "fingerprint cloning", "AEPS fraud"],
    "Cyber Stalking": ["cyber stalking", "cyber bullying", "online harassment", "online stalking", "digital harassment"],
}
MAX_PAGES = 3

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8"
}
BASE_URL = "https://www.consumercomplaints.in"

OUTPUT_CSV = "data/consumer_complaints.csv"
OUTPUT_JSON = "data/consumer_complaints.json"

def classify_narrative_type(text):
    text_lower = text.lower()
    if "my account" in text_lower or "i lost" in text_lower or "duped me" in text_lower:
        return "VICTIM"
    elif "tried to" in text_lower or "almost" in text_lower or "did not share" in text_lower:
        return "NEAR-MISS"
    return "THIRD-PARTY"

In [ ]:
# 1.5 Subcategory to Category Mapping
CATEGORY_MAPPING = {
    "General Cybercrime": "General Cybercrime / Cyber Fraud Terms",
    "Cyber Scam": "General Cybercrime / Cyber Fraud Terms",
    "Financial Cyber Fraud": "General Cybercrime / Cyber Fraud Terms",
    "UPI Fraud": "UPI and Digital Payment Fraud",
    "QR Code Fraud": "UPI and Digital Payment Fraud",
    "Payment Link Fraud": "UPI and Digital Payment Fraud",
    "Mobile Wallet Fraud": "UPI and Digital Payment Fraud",
    "OTP Fraud": "OTP and Authentication Fraud",
    "SIM Swap": "OTP and Authentication Fraud",
    "KYC Fraud": "OTP and Authentication Fraud",
    "Digital Arrest": "Digital Arrest Scam",
    "Impersonation Scam": "Digital Arrest Scam",
    "Video Call Coercion": "Digital Arrest Scam",
    "Phishing": "Phishing, Vishing, and Smishing",
    "Vishing": "Phishing, Vishing, and Smishing",
    "Smishing": "Phishing, Vishing, and Smishing",
    "Loan App Fraud": "Online Lending and Loan App Fraud",
    "Loan App Extortion": "Online Lending and Loan App Fraud",
    "Investment Scam": "Investment and Trading Fraud",
    "Stock Market Fraud": "Investment and Trading Fraud",
    "Task Scam": "Investment and Trading Fraud",
    "Identity Theft": "Identity Theft and Data Breach",
    "Data Breach": "Identity Theft and Data Breach",
    "Social Engineering": "Social Engineering and Romance/Sextortion",
    "Romance Scam": "Social Engineering and Romance/Sextortion",
    "Sextortion": "Social Engineering and Romance/Sextortion",
    "E-Commerce Fraud": "E-Commerce and Delivery Fraud",
    "Delivery Fraud": "E-Commerce and Delivery Fraud",
    "Ransomware": "Ransomware and Malware",
    "Banking Malware": "Ransomware and Malware",
    "Deepfake Fraud": "Emerging and Miscellaneous Fraud Types",
    "Utility Scam": "Emerging and Miscellaneous Fraud Types",
    "Aadhaar Fraud": "Emerging and Miscellaneous Fraud Types",
    "Cyber Stalking": "Emerging and Miscellaneous Fraud Types"
}

In [4]:
# 2. Scraping Functions
def scrape_complaint_details(url):
    """
    Fetches the full text of a complaint from its specific dynamic page.
    """
    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        if response.status_code != 200: 
            return None
            
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # In consumercomplaints.in, the review text uses schema.org itemprop tags
        review_body = soup.find('div', itemprop='reviewBody')
        if review_body:
            # We must strip out inner tables so we don't grab raw table metadata inside text representation
            for block in review_body.find_all(['table']):
                block.decompose()
            return review_body.get_text(separator=' ', strip=True)
            
        # Fallback to td if schema items aren't directly available
        td_complaints = soup.find_all('td', class_='complaint')
        if len(td_complaints) > 0:
            return td_complaints[-1].get_text(separator=' ', strip=True)
        
        return ""
    except Exception as e:
        print(f"Error fetching {url}: {e}")
        return ""

def search_and_scrape(keyword, category, max_pages=3):
    """
    Searches keyword across paginated pages. Extracts the base items.
    """
    complaints = []
    seen_urls = set()
    
    for page in range(1, max_pages + 1):
        # Apply rate limiting politely before each fetch
        time.sleep(random.uniform(2.0, 5.0))
        
        print(f"[*] Scraping '{keyword}' ({category}) - Page {page}")
        # Build search URI
        search_url = f"{BASE_URL}/?search={keyword.replace(' ', '+')}&page={page}"
        
        try:
            response = requests.get(search_url, headers=HEADERS, timeout=15)
            if response.status_code != 200: 
                print(f"Failed to fetch page {page}. Status code: {response.status_code}")
                break
            
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # Identify listing objects (verified class from devtools dump)
            results = soup.find_all('div', class_='complaint-box-results')
                
            print(f"Found {len(results)} complaint blocks on page {page}.")
            
            if not results:
                # Could mean either 0 matches exist, or we hit an actual block page.
                print("No results found on this page. Stopping pagination for this keyword.")
                break
                
            for item in results:
                # Title info
                title_tag = item.find('a', class_='complaint-box-results__title')
                if not title_tag: continue
                
                title = title_tag.get_text(separator=' ', strip=True)
                href = title_tag['href']
                url = BASE_URL + href if href.startswith('/') else href
                
                # Check for duplicates earlier before requesting
                if url in seen_urls: continue
                seen_urls.add(url)
                
                # Metadata: date
                date_tag = item.find('div', class_='author-box__date')
                original_date = date_tag.get_text(strip=True) if date_tag else "Unknown Date"
                
                # Metadata: Username
                user_tag = item.find('div', class_='author-box__user')
                username = user_tag.get_text(separator=' ', strip=True) if user_tag else "Unknown User"
                
                # Metadata: Company string often lies in the strong tag portion of the title heading
                company_tag = title_tag.find('strong', class_='complaint-box-results__title-strong')
                company = "Unknown"
                if company_tag:
                    company_val = company_tag.get_text(strip=True).split('/')[0].strip()
                    if company_val:
                        company = company_val
                
                complaints.append({
                    "Search Query Used": keyword,
                    "Category": category,
                    "Title/Headline": title,
                    "URL": url,
                    "Original Date": original_date,
                    "Company": company,
                    "Username": username,
                    "Full Text": "" # Placeholder for detailed fetch stage
                })
                
        except Exception as e:
            print(f"Error on page {page} for '{keyword}': {e}")
        
    return complaints

In [5]:
# 3. Execute Scraping and Format Output
import os

all_data = []

for sub_category, keywords in KEYWORD_TAXONOMY.items():
    for kw in keywords:
        data = search_and_scrape(kw, sub_category, max_pages=MAX_PAGES)
        all_data.extend(data)

print(f"\nTotal pre-filtered complaints scraped: {len(all_data)}")

# Load existing JSON for deduplication to avoid pulling data again
existing_json_data = []
existing_urls = set()
max_id = 0

if os.path.exists(OUTPUT_JSON):
    try:
        with open(OUTPUT_JSON, "r", encoding="utf-8") as f:
            existing_json_data = json.load(f)
            for item in existing_json_data:
                # We also need to get existing URLs to skip full fetch for duplicates
                url = item.get("URL")
                if url:
                    existing_urls.add(url)
                # Find max ID
                if "StructuredData" in item and "Unique ID" in item["StructuredData"]:
                    uid_str = item["StructuredData"]["Unique ID"]
                    try:
                        uid_num = int(uid_str.split('-')[-1])
                        if uid_num > max_id:
                            max_id = uid_num
                    except:
                        pass
    except Exception as e:
        print(f"Error loading existing JSON: {e}")

# Deduplicate new data by URL
unique_new_data = {}
for v in all_data:
    if v["URL"] not in existing_urls:
        if v["URL"] not in unique_new_data:
            unique_new_data[v["URL"]] = v
new_all_data = list(unique_new_data.values())

print(f"Found {len(new_all_data)} new unique complaints to process (skipped {len(all_data) - len(new_all_data)} duplicates).")

# Format data to required schema
formatted_records = []
today_date = datetime.now().strftime("%Y-%m-%d")

next_id = max_id + 1

for idx, item in enumerate(new_all_data, start=1):
    url = item["URL"]
    print(f"[{idx}/{len(new_all_data)}] Fetching full text for {url}")
    
    # 5. Visit each complaint URL to extract full text and rate-limit each visit!
    time.sleep(random.uniform(2.0, 4.0)) 
    full_text = scrape_complaint_details(url)
    
    # Fallback assignment
    if not full_text:
        full_text = item["Title/Headline"]

    item["Full Text"] = full_text
    
    # 6. Map extracted data into structured format
    sub_category = item["Category"]
    parent_category = CATEGORY_MAPPING.get(sub_category, "Unknown Category")
    
    record = {
        "Unique ID": f"NA-{next_id:04d}",
        "Date of Collection": today_date,
        "Collector Name": "Soubhik Sarkar",
        "Source Platform": "ConsumerComplaints.in",
        "Source Publication": "ConsumerComplaints",
        "Original Date": item["Original Date"],
        "Title/Headline": item["Title/Headline"],
        "URL": item["URL"],
        "Search Query Used": item["Search Query Used"],
        "Fraud Category": parent_category,
        "Fraud Subcategory": sub_category,
        "Narrative Type": classify_narrative_type(full_text),
        "Full Text Saved": "Yes",
        "Full Text": full_text,  # Added actual post text here
        "Notes": f"Company: {item['Company']}, Username: {item.get('Username', 'Unknown')}"
    }
    next_id += 1
    
    # Save formatted details alongside raw object
    item["StructuredData"] = record
    formatted_records.append(record)

# Store dataset as dataframe to view top elements briefly
if formatted_records:
    df_new = pd.DataFrame(formatted_records)
    display(df_new.head())
else:
    df_new = pd.DataFrame()
    print("No new records to display.")

[*] Scraping 'cyber crime' (General Cybercrime) - Page 1
Found 25 complaint blocks on page 1.
[*] Scraping 'cyber crime' (General Cybercrime) - Page 2
Found 25 complaint blocks on page 2.
[*] Scraping 'cyber crime' (General Cybercrime) - Page 3
Found 25 complaint blocks on page 3.
[*] Scraping 'cybercrime' (General Cybercrime) - Page 1
Found 25 complaint blocks on page 1.
[*] Scraping 'cybercrime' (General Cybercrime) - Page 2
Found 25 complaint blocks on page 2.
[*] Scraping 'cybercrime' (General Cybercrime) - Page 3
Found 25 complaint blocks on page 3.
[*] Scraping 'cyber fraud' (General Cybercrime) - Page 1
Found 25 complaint blocks on page 1.
[*] Scraping 'cyber fraud' (General Cybercrime) - Page 2
Found 25 complaint blocks on page 2.
[*] Scraping 'cyber fraud' (General Cybercrime) - Page 3
Found 25 complaint blocks on page 3.
[*] Scraping 'online fraud' (General Cybercrime) - Page 1
Found 25 complaint blocks on page 1.
[*] Scraping 'online fraud' (General Cybercrime) - Page 2
Foun

,Unique ID,Date of Collection,Collector Name,Source Platform,Source Publication,Original Date,Title/Headline,URL,Search Query Used,Fraud Category,Fraud Subcategory,Narrative Type,Full Text Saved,Notes
0,NA-0278,2026-03-28,Soubhik Sarkar,ConsumerComplaints.in,ConsumerComplaints,"May 19, 2025",Police-Tamil Nadu-Madurai City- Cyber Crime Ce...,https://www.consumercomplaints.in/police-tamil...,cyber crime,General Cybercrime,Unknown,VICTIM,Yes,Company: Police-Tamil Nadu-Madurai City-Cyber ...
1,NA-0279,2026-03-28,Soubhik Sarkar,ConsumerComplaints.in,ConsumerComplaints,Unknown Date,Cyber Crime — Regarding money lost,https://www.consumercomplaints.in/bycompany/cy...,cyber crime,General Cybercrime,Unknown,VICTIM,Yes,"Company: Cyber Crime, Username: AmanTripathi"
2,NA-0280,2026-03-28,Soubhik Sarkar,ConsumerComplaints.in,ConsumerComplaints,Unknown Date,Police-Tamil Nadu-Madurai City- Cyber Crime Ce...,https://www.consumercomplaints.in/bycompany/po...,cyber crime,General Cybercrime,Unknown,VICTIM,Yes,Company: Police-Tamil Nadu-Madurai City-Cyber ...
3,NA-0281,2026-03-28,Soubhik Sarkar,ConsumerComplaints.in,ConsumerComplaints,Unknown Date,Cyber Crime — Fraud job / online money investm...,https://www.consumercomplaints.in/bycompany/cy...,cyber crime,General Cybercrime,Unknown,VICTIM,Yes,"Company: Cyber Crime, Username: kpreeti"
4,NA-0282,2026-03-28,Soubhik Sarkar,ConsumerComplaints.in,ConsumerComplaints,Unknown Date,Cyber Crime — Bank account freeze due to cyber...,https://www.consumercomplaints.in/bycompany/cy...,cyber crime,General Cybercrime,Unknown,VICTIM,Yes,"Company: Cyber Crime, Username: Ruchi1233"


In [6]:
# 4. Save to CSV and JSON
import os
os.makedirs("data", exist_ok=True)

new_records_count = len(formatted_records)
total_records_count = len(existing_json_data)

if new_records_count > 0:
    # Append to CSV
    if os.path.exists(OUTPUT_CSV):
        df_existing = pd.read_csv(OUTPUT_CSV)
        df_combined = pd.concat([df_existing, df_new], ignore_index=True)
        # Assuming duplicate URL means duplicate record
        df_combined.drop_duplicates(subset=["URL"], keep="last", inplace=True)
    else:
        df_combined = df_new
        
    df_combined.to_csv(OUTPUT_CSV, index=False)
    
    # Append to JSON
    # we already loaded existing_json_data
    existing_json_data.extend(new_all_data)
    # Deduplicate JSON again just in case
    json_dedup = {v["URL"]: v for v in existing_json_data}
    final_json_data = list(json_dedup.values())
    
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(final_json_data, f, indent=4, ensure_ascii=False)
        
    total_records_count = len(final_json_data)
    print(f"Saved {new_records_count} new records. Updated files.")

else:
    print("No new records to save.")

# Add a progress summary print at the end of the run
duplicates_skipped = len(all_data) - len(new_all_data)
print("\n--- Summary ---")
print(f"Records scraped this session: {len(all_data)}")
print(f"Duplicates skipped: {duplicates_skipped}")
print(f"New records added: {new_records_count}")
print(f"Total records now in dataset: {total_records_count}")

Saved 4041 new records. Updated files.

--- Summary ---
Records scraped this session: 4494
Duplicates skipped: 453
New records added: 4041
Total records now in dataset: 4318


In [10]:
import pandas as pd
import json
import os

# 1. Structured Taxonomy Definition based on specific instructions
TAXONOMY_STRUCTURE = {
    "General Cybercrime / Cyber Fraud Terms": {
        "General Cybercrime": ["cyber crime", "cybercrime", "cyber fraud", "online fraud", "internet fraud", "digital fraud"],
        "Cyber Scam": ["cyber scam", "online scam", "internet scam", "online cheating", "online loot"],
        "Financial Cyber Fraud": ["financial fraud online", "net banking fraud", "e-banking fraud"]
    },
    "UPI and Digital Payment Fraud": {
        "UPI Fraud": ["UPI fraud", "UPI scam", "Google Pay fraud", "PhonePe fraud", "Paytm fraud", "BHIM fraud"],
        "QR Code Fraud": ["QR code scam", "QR code fraud", "scan and pay fraud", "fake QR code"],
        "Payment Link Fraud": ["payment link fraud", "collect request scam", "UPI collect fraud", "fake payment link"],
        "Mobile Wallet Fraud": ["mobile wallet fraud", "e-wallet scam", "digital wallet fraud"]
    },
    "OTP and Authentication Fraud": {
        "OTP Fraud": ["OTP fraud", "OTP scam", "OTP theft", "OTP sharing scam"],
        "SIM Swap": ["SIM swap fraud", "SIM cloning", "SIM swap scam", "duplicate SIM fraud"],
        "KYC Fraud": ["KYC fraud", "KYC scam", "fake KYC", "Aadhaar KYC scam"]
    },
    "Digital Arrest Scam": {
        "Digital Arrest": ["digital arrest", "digital arrest scam", "fake arrest", "video call arrest"],
        "Impersonation Scam": ["police impersonation scam", "CBI fraud call", "customs fraud call", "TRAI scam call", "ED scam call", "fake police call"],
        "Video Call Coercion": ["video call scam", "video call blackmail", "video call extortion", "Skype arrest", "fake interrogation"]
    },
    "Phishing, Vishing, and Smishing": {
        "Phishing": ["phishing", "phishing attack", "phishing email", "fake website", "spoof website"],
        "Vishing": ["vishing", "voice phishing", "phone scam", "fraud call", "fake bank call"],
        "Smishing": ["smishing", "SMS fraud", "SMS scam", "fake SMS", "phishing SMS"]
    },
    "Online Lending and Loan App Fraud": {
        "Loan App Fraud": ["loan app fraud", "instant loan scam", "loan app harassment", "illegal loan app", "Chinese loan app"],
        "Loan App Extortion": ["loan app blackmail", "loan app threat", "morphed photos loan", "contact list harassment", "recovery agent threat"]
    },
    "Investment and Trading Fraud": {
        "Investment Scam": ["investment scam", "Ponzi scheme", "online investment fraud", "crypto scam", "bitcoin fraud", "forex trading scam", "pig butchering"],
        "Stock Market Fraud": ["stock market scam", "share trading fraud", "demat fraud", "pump and dump", "trading app scam"],
        "Task Scam": ["task fraud", "part time job scam", "work from home scam", "Telegram task scam", "like and earn", "review task fraud"]
    },
    "Identity Theft and Data Breach": {
        "Identity Theft": ["identity theft", "identity fraud", "impersonation fraud", "Aadhaar misuse", "PAN fraud"],
        "Data Breach": ["data breach", "data leak", "data theft", "personal data leak", "customer data breach"]
    },
    "Social Engineering and Romance/Sextortion": {
        "Social Engineering": ["social engineering attack", "social engineering fraud", "manipulation scam", "trust scam"],
        "Romance Scam": ["romance scam", "dating fraud", "matrimonial fraud", "love scam", "honey trap", "catfishing fraud"],
        "Sextortion": ["sextortion", "sexual blackmail online", "webcam blackmail", "nude video blackmail"]
    },
    "E-Commerce and Delivery Fraud": {
        "E-Commerce Fraud": ["e-commerce fraud", "online shopping fraud", "fake product scam", "Flipkart fraud", "Amazon fraud", "refund scam"],
        "Delivery Fraud": ["fake delivery", "courier fraud", "customs duty scam", "parcel scam", "FedEx scam", "delivery OTP scam"]
    },
    "Ransomware and Malware": {
        "Ransomware": ["ransomware attack", "cyber ransom", "data encryption attack", "file lock malware"],
        "Banking Malware": ["banking trojan", "banking malware", "mobile banking virus", "keylogger fraud", "AnyDesk fraud", "TeamViewer scam", "screen sharing scam"]
    },
    "Emerging and Miscellaneous Fraud Types": {
        "Deepfake Fraud": ["deepfake scam", "deepfake fraud", "AI voice scam", "voice cloning scam", "fake video call"],
        "Utility Scam": ["electricity bill scam", "utility fraud", "disconnection scam", "fake bill", "power cut threat"],
        "Aadhaar Fraud": ["Aadhaar fraud", "Aadhaar scam", "biometric fraud", "fingerprint cloning", "AEPS fraud"],
        "Cyber Stalking": ["cyber stalking", "cyber bullying", "online harassment", "online stalking", "digital harassment"]
    }
}

# 2. Build reverse maps for fast O(1) lookups
query_to_subcat_strict = {}
subcat_to_cat_strict = {}

for parent_category, subcategories in TAXONOMY_STRUCTURE.items():
    for subcat, queries in subcategories.items():
        subcat_to_cat_strict[subcat] = parent_category
        for q in queries:
            # Map case-insensitive search query to its correct subcategory
            query_to_subcat_strict[q.lower()] = subcat

OUTPUT_CSV = "data/consumer_complaints.csv"
OUTPUT_JSON = "data/consumer_complaints.json"

def process_query(query):
    """Returns correct (Category, Subcategory) tuple based on query."""
    if not isinstance(query, str):
        return "Unknown", "Unknown"
        
    query_clean = query.strip().lower()
    
    # Precise match mapping
    subcat = query_to_subcat_strict.get(query_clean, "Unknown")
    cat = subcat_to_cat_strict.get(subcat, "Unknown")
    
    return cat, subcat

# -- Apply to CSV --
print("Updating CSV...")
if os.path.exists(OUTPUT_CSV):
    try:
        df = pd.read_csv(OUTPUT_CSV)
        
        # Apply the logic row by row based purely on "Search Query Used"
        for index, row in df.iterrows():
            q_used = row.get("Search Query Used", "")
            correct_cat, correct_subcat = process_query(q_used)
            
            df.at[index, "Fraud Category"] = correct_cat
            df.at[index, "Fraud Subcategory"] = correct_subcat
            
        df.to_csv(OUTPUT_CSV, index=False)
        print(f"✅ CSV updated successfully. ({len(df)} records fixed)")
        
        # Display sample to verify
        display(df[["Search Query Used", "Fraud Category", "Fraud Subcategory"]].head(10))
    except PermissionError:
         print("❌ Permission Error: The CSV is likely open in Excel. Please close it and rerun this cell.")
else:
    print("❌ CSV not found.")

# -- Apply to JSON --
print("\nUpdating JSON...")
if os.path.exists(OUTPUT_JSON):
    try:
        with open(OUTPUT_JSON, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        for item in data:
            q_used = item.get("Search Query Used", "")
            correct_cat, correct_subcat = process_query(q_used)
            
            item["Category"] = correct_subcat 
            if "StructuredData" in item:
                item["StructuredData"]["Fraud Category"] = correct_cat
                item["StructuredData"]["Fraud Subcategory"] = correct_subcat

        with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=4, ensure_ascii=False)
        print(f"✅ JSON updated successfully. ({len(data)} records fixed)")
    except Exception as e:
         print(f"❌ Error saving JSON: {e}")
else:
    print("❌ JSON not found.")

Updating CSV...
✅ CSV updated successfully. (4318 records fixed)


,Search Query Used,Fraud Category,Fraud Subcategory
0,UPI fraud,UPI and Digital Payment Fraud,UPI Fraud
1,UPI fraud,UPI and Digital Payment Fraud,UPI Fraud
2,UPI fraud,UPI and Digital Payment Fraud,UPI Fraud
3,UPI fraud,UPI and Digital Payment Fraud,UPI Fraud
4,UPI fraud,UPI and Digital Payment Fraud,UPI Fraud
5,UPI fraud,UPI and Digital Payment Fraud,UPI Fraud
6,UPI fraud,UPI and Digital Payment Fraud,UPI Fraud
7,UPI fraud,UPI and Digital Payment Fraud,UPI Fraud
8,UPI fraud,UPI and Digital Payment Fraud,UPI Fraud
9,UPI fraud,UPI and Digital Payment Fraud,UPI Fraud



Updating JSON...
✅ JSON updated successfully. (4318 records fixed)


In [14]:
# --- POST-PROCESSING: EXTRACT TEXT TO TXT FILES AND CLEAN UP CSV/JSON ---
# Run this standalone cell to process the saved records
import os
import json
import pandas as pd

CSV_FILE = "data/consumer_complaints.csv"
JSON_FILE = "data/consumer_complaints.json"
TXT_DIR = "data/txt"

os.makedirs(TXT_DIR, exist_ok=True)

try:
    print("Loading existing JSON data...")
    with open(JSON_FILE, "r", encoding="utf-8") as f:
        json_data = json.load(f)
        
    print(f"Extracting full text and saving as .txt files in {TXT_DIR}...")
    url_to_txt_filename = {}
    
    # 1. Process JSON saving txt files and updating the structure
    for item in json_data:
        struct_data = item.get("StructuredData", {})
        
        # Determine Unique ID
        unique_id = struct_data.get("Unique ID") or item.get("Unique ID")
        if not unique_id:
            # Generate a temporary one if completely missing
            url_hash = abs(hash(item.get("URL", ""))) % 100000
            unique_id = f"NA-temp-{url_hash}"
            
        txt_filename = f"{unique_id}.txt"
        
        # Find the full text
        full_text = item.get("Full Text") or struct_data.get("Full Text") or item.get("Title/Headline") or ""
        
        # Save to txt file
        with open(os.path.join(TXT_DIR, txt_filename), "w", encoding="utf-8") as f_out:
            f_out.write(full_text)
            
        # Store for CSV lookup
        url = item.get("URL") or struct_data.get("URL")
        if url:
            url_to_txt_filename[url] = txt_filename
            
        # 2. Re-structure JSON element: Add "TXT File Name" and remove "Full Text"
        struct_data["TXT File Name"] = txt_filename
        
        if "Full Text" in struct_data:
            del struct_data["Full Text"]
        if "Full Text Saved" in struct_data:
            del struct_data["Full Text Saved"]
            
        if "Full Text" in item:
            del item["Full Text"]
            
        item["StructuredData"] = struct_data

    # 3. Save cleaned up JSON
    print("Cleaning up JSON file (removing full text blobs)...")
    with open(JSON_FILE, "w", encoding="utf-8") as f:
        json.dump(json_data, f, indent=4, ensure_ascii=False)
        
    # 4. Clean up and Update CSV
    print("Loading CSV...")
    if os.path.exists(CSV_FILE):
        df = pd.read_csv(CSV_FILE)
        
        # Apply the TXT File Name
        df["TXT File Name"] = df["URL"].apply(lambda u: url_to_txt_filename.get(u, ""))
        
        # Remove old columns
        if "Full Text" in df.columns:
            df.drop(columns=["Full Text"], inplace=True)
        if "Full Text Saved" in df.columns:
            df.drop(columns=["Full Text Saved"], inplace=True)
            
        # Reorder to put TXT File Name alongside Notes realistically
        cols = list(df.columns)
        if "TXT File Name" in cols and "Notes" in cols:
            cols.remove("TXT File Name")
            notes_idx = cols.index("Notes")
            cols.insert(notes_idx, "TXT File Name")
            df = df[cols]
            
        print(f"Saving updated data to {CSV_FILE}...")
        df.to_csv(CSV_FILE, index=False, encoding="utf-8-sig")
        
        # Also save an Excel version since you requested it!
        excel_file = "data/consumer_complaints.xlsx"
        print(f"Saving a clean Excel version to {excel_file}...")
        df.to_excel(excel_file, index=False)
        
        print(f"\n✅ Success! Processed {len(json_data)} records.")
        print("1. All post texts are now saved individually in 'data/txt/'.")
        print("2. The CSV and JSON files have been cleaned of large text blocks.")
        print("3. A shiny new 'consumer_complaints.xlsx' file has been created for you.")
    else:
        print("CSV not found.")
        
except PermissionError:
    print("\n❌ Permission Denied! Please close Excel if you have the CSV open and run this cell again.")
except Exception as e:
    print(f"\n❌ An error occurred: {e}")

Loading existing JSON data...
Extracting full text and saving as .txt files in data/txt...
Cleaning up JSON file (removing full text blobs)...
Loading CSV...
Saving updated data to data/consumer_complaints.csv...
Saving a clean Excel version to data/consumer_complaints.xlsx...

✅ Success! Processed 4318 records.
1. All post texts are now saved individually in 'data/txt/'.
2. The CSV and JSON files have been cleaned of large text blocks.
3. A shiny new 'consumer_complaints.xlsx' file has been created for you.


In [18]:
# === NEW DIAGNOSTIC CELL (ANCHOR FOCUSED) ===
# Diagnostic — paste any failing URL here
import requests
from bs4 import BeautifulSoup
import time

test_url = "https://www.consumercomplaints.in/hdfc-bank-limited-b100164/page/223#get-cl2940692"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8"
}

print(f"Fetching diagnostic page: {test_url}")
response = requests.get(test_url, headers=headers)
time.sleep(3)
soup = BeautifulSoup(response.text, 'html.parser')

print("=== All elements with IDs containing 'cl' ===")
for tag in soup.find_all(id=True):
    if 'cl' in str(tag.get('id', '')).lower():
        print(f"id={tag.get('id')} | tag={tag.name}")
        print(f"preview: {tag.get_text(strip=True)[:200]}")
        print("---")

Fetching diagnostic page: https://www.consumercomplaints.in/hdfc-bank-limited-b100164/page/223#get-cl2940692
=== All elements with IDs containing 'cl' ===


In [19]:
# === UPDATED SCRAPING FUNCTION FOR FUTURE RUNS ===
# We are redefining it entirely down here so previous cells are untouched. 
# It handles anchor-based URLs to jump to the right fragment inside listing pages.
import random
import time

def scrape_complaint_details(url):
    """
    Fetches the full text of a complaint from its specific dynamic page or anchor fragment.
    """
    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        if response.status_code != 200: 
            return None
            
        time.sleep(random.uniform(2, 4))
        soup = BeautifulSoup(response.text, 'html.parser')

        # Extract the anchor fragment (e.g. "get-cl2940692" -> "cl2940692")
        anchor = ""
        if "#" in url:
            anchor = url.split("#")[-1]  # e.g. "get-cl2940692"
            complaint_id = anchor.replace("get-", "")  # e.g. "cl2940692"
        else:
            complaint_id = None

        # 1. First approach: If we have a complaint ID, find the element directly
        if complaint_id:
            # Try finding by id attribute directly
            el = soup.find(id=complaint_id) or soup.find(id=f"get-{complaint_id}")
            if el:
                # Strip tables if present
                for block in el.find_all(['table']):
                    block.decompose()
                text = el.get_text(separator='\n', strip=True)
                if len(text) > 50:
                    return text

            # Try finding a div/section that contains this complaint ID as a data attribute
            el = soup.find(attrs={"data-id": complaint_id})
            if el:
                for block in el.find_all(['table']):
                    block.decompose()
                text = el.get_text(separator='\n', strip=True)
                if len(text) > 50:
                    return text

        # 2. Fallback: try common generic layout selectors
        selectors = [
            ('div', {'itemprop': 'reviewBody'}),
            ('div', {'class': 'complaint-text'}),
            ('div', {'class': 'complaint-detail'}),
            ('div', {'class': 'complaint-body'}),
            ('div', {'class': 'complaint-content'}),
            ('td', {'class': 'complaint'}),
            ('div', {'itemprop': 'description'}),
            ('div', {'class': 'description'}),
        ]

        for tag, attrs in selectors:
            el = soup.find(tag, attrs)
            if el:
                for block in el.find_all(['table']):
                    block.decompose()
                text = el.get_text(separator='\n', strip=True)
                if len(text) > 50:
                    return text

        # Diagnostic print so we can see the real structure to debug
        print(f"⚠️ Extraction failed for {url}")
        print("--- Printing all IDs found on page ---")
        for tag in soup.find_all(id=True):
            if 'cl' in str(tag.get('id', '')).lower():
                print(f"  id={tag.get('id')} | tag={tag.name} | preview={tag.get_text(strip=True)[:100]}")
        return ""

    except Exception as e:
        print(f"Error fetching {url}: {e}")
        return ""

In [20]:
# === UPDATED POST-PROCESSING: EXTRACT TEXT TO TXT FILES ===
# Run this standalone cell to process the saved records (incorporating the clean [EXTRACTION FAILED] logic)
import os
import json
import pandas as pd

CSV_FILE = "data/consumer_complaints.csv"
JSON_FILE = "data/consumer_complaints.json"
TXT_DIR = "data/txt"

os.makedirs(TXT_DIR, exist_ok=True)

try:
    print("Loading existing JSON data...")
    with open(JSON_FILE, "r", encoding="utf-8") as f:
        json_data = json.load(f)
        
    print(f"Extracting full text and saving as .txt files in {TXT_DIR}...")
    url_to_txt_filename = {}
    
    # 1. Process JSON saving txt files and updating the structure
    for item in json_data:
        struct_data = item.get("StructuredData", {})
        
        # Determine Unique ID
        unique_id = struct_data.get("Unique ID") or item.get("Unique ID")
        if not unique_id:
            url_hash = abs(hash(item.get("URL", ""))) % 100000
            unique_id = f"NA-temp-{url_hash}"
            
        txt_filename = f"{unique_id}.txt"
        
        # Apply the new fallback logic
        title = item.get("Title/Headline", "")
        # Get whatever was historically saved as full text
        full_text = item.get("Full Text", "") or struct_data.get("Full Text", "")
        if isinstance(full_text, str):
            full_text = full_text.strip()
            
        # Clean flag if extraction failed or was overwritten by just the title fallback without the URL inside the text file
        if not full_text or full_text == title:
            full_text = f"[EXTRACTION FAILED — please visit the URL manually]\n\n{title}"
        
        # Save to txt file
        with open(os.path.join(TXT_DIR, txt_filename), "w", encoding="utf-8") as f_out:
            f_out.write(full_text)
            
        # Store for CSV lookup
        url = item.get("URL") or struct_data.get("URL")
        if url:
            url_to_txt_filename[url] = txt_filename
            
        # 2. Re-structure JSON element: Add "TXT File Name" and remove "Full Text"
        struct_data["TXT File Name"] = txt_filename
        
        if "Full Text" in struct_data:
            del struct_data["Full Text"]
        if "Full Text Saved" in struct_data:
            del struct_data["Full Text Saved"]
        if "Full Text" in item:
            del item["Full Text"]
            
        item["StructuredData"] = struct_data

    # 3. Save cleaned up JSON
    print("Cleaning up JSON file (removing full text blobs)...")
    with open(JSON_FILE, "w", encoding="utf-8") as f:
        json.dump(json_data, f, indent=4, ensure_ascii=False)
        
    # 4. Clean up and Update CSV
    print("Loading CSV...")
    if os.path.exists(CSV_FILE):
        df = pd.read_csv(CSV_FILE)
        
        # Apply the TXT File Name
        df["TXT File Name"] = df["URL"].apply(lambda u: url_to_txt_filename.get(u, ""))
        
        # Remove old columns
        if "Full Text" in df.columns:
            df.drop(columns=["Full Text"], inplace=True)
        if "Full Text Saved" in df.columns:
            df.drop(columns=["Full Text Saved"], inplace=True)
            
        cols = list(df.columns)
        if "TXT File Name" in cols and "Notes" in cols:
            cols.remove("TXT File Name")
            notes_idx = cols.index("Notes")
            cols.insert(notes_idx, "TXT File Name")
            df = df[cols]
            
        print(f"Saving updated data to {CSV_FILE}...")
        df.to_csv(CSV_FILE, index=False, encoding="utf-8-sig")
        
        excel_file = "data/consumer_complaints.xlsx"
        print(f"Saving a clean Excel version to {excel_file}...")
        df.to_excel(excel_file, index=False)
        
        print(f"\n✅ Success! Processed {len(json_data)} records.")
        print("1. All post texts are now saved individually in 'data/txt/'. Missing texts are cleanly flagged.")
    else:
        print("CSV not found.")
        
except PermissionError:
    print("\n❌ Permission Denied! Please close Excel if you have the CSV open and run this cell again.")
except Exception as e:
    print(f"\n❌ An error occurred: {e}")

Loading existing JSON data...
Extracting full text and saving as .txt files in data/txt...
Cleaning up JSON file (removing full text blobs)...
Loading CSV...
Saving updated data to data/consumer_complaints.csv...
Saving a clean Excel version to data/consumer_complaints.xlsx...

✅ Success! Processed 4318 records.
1. All post texts are now saved individually in 'data/txt/'. Missing texts are cleanly flagged.


In [25]:
import json

# Try loading the backup
with open("data/consumer_complaints_backup.json", "r", encoding="utf-8-sig") as f:
    raw = f.read()

# Clean common notepad encoding issues
raw = raw.strip()
if not raw.startswith('['):
    raw = '[' + raw
if not raw.endswith(']'):
    raw = raw + ']'

try:
    data = json.loads(raw)
    print(f"✅ Loaded successfully: {len(data)} records")
    print(f"Full Text sample: {data[0].get('Full Text', '❌ GONE')[:200]}")
except json.JSONDecodeError as e:
    print(f"❌ JSON error at: {e}")
    # Show the problematic area
    lines = raw.split('\n')
    print(f"Around line {e.lineno}:")
    print('\n'.join(lines[max(0, e.lineno-3):e.lineno+3]))

❌ JSON error at: Invalid control character at: line 25048 column 39 (char 2196532)
Around line 25048:
        "Company": "Skill Lync",
        "Username": "surajchaurasia",
        "Full Text": "Dear Sir/Madam, 
I’m Suraj Chaurasia from Lucknow, Uttar Pradesh. I am writing this to bring attention on the currently on-going scam in India and seeking justice regarding a fraudulent involving Skill-Lync and their loan partner Eduvnz. I am a victim for their scam. On October month 2022 I got a call from Mr. Sidharth Dubey from skill lync and they introduced me a online course as a PG programm on first call I rejected their course after that weekly I started getting 3 to 4 calls from them and they started explaining about their course trends and future, at that time I’m Jobless, I told my family situation where I can’t afford that course because we are not financially stable where I”m also studying. I kept on rejecting their offers whenever I got a call from them. At starting they offered me 

In [26]:
import json
import re

with open("data/consumer_complaints_backup.json", "r", encoding="utf-8-sig") as f:
    raw = f.read()

# Remove all invalid control characters except normal ones like \n \t
raw = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', raw)

# Try loading now
try:
    data = json.loads(raw)
    print(f"✅ Loaded successfully: {len(data)} records")
    print(f"Sample Full Text: {data[0].get('Full Text', '❌ GONE')[:200]}")
    
    # Save the fixed version as the main json immediately
    with open("data/consumer_complaints.json", "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)
    print("✅ Fixed JSON saved as consumer_complaints.json")
    
except json.JSONDecodeError as e:
    print(f"❌ Still failing at: {e}")
    lines = raw.split('\n')
    print('\n'.join(lines[max(0, e.lineno-3):e.lineno+3]))

❌ Still failing at: Invalid control character at: line 25048 column 39 (char 2196532)
        "Company": "Skill Lync",
        "Username": "surajchaurasia",
        "Full Text": "Dear Sir/Madam, 
I’m Suraj Chaurasia from Lucknow, Uttar Pradesh. I am writing this to bring attention on the currently on-going scam in India and seeking justice regarding a fraudulent involving Skill-Lync and their loan partner Eduvnz. I am a victim for their scam. On October month 2022 I got a call from Mr. Sidharth Dubey from skill lync and they introduced me a online course as a PG programm on first call I rejected their course after that weekly I started getting 3 to 4 calls from them and they started explaining about their course trends and future, at that time I’m Jobless, I told my family situation where I can’t afford that course because we are not financially stable where I”m also studying. I kept on rejecting their offers whenever I got a call from them. At starting they offered me a course for Rs.

In [27]:
import json
import re

with open("data/consumer_complaints_backup.json", "r", encoding="utf-8-sig") as f:
    raw = f.read()

# Remove invalid control characters
raw = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', raw)

# Fix unescaped newlines inside JSON strings
# This replaces actual newlines inside string values with \n
def fix_newlines_in_strings(s):
    result = []
    in_string = False
    escape = False
    for char in s:
        if escape:
            result.append(char)
            escape = False
        elif char == '\\':
            result.append(char)
            escape = True
        elif char == '"' and not escape:
            in_string = not in_string
            result.append(char)
        elif in_string and char == '\n':
            result.append('\\n')  # replace real newline with escaped \n
        elif in_string and char == '\r':
            result.append('\\r')
        else:
            result.append(char)
    return ''.join(result)

raw = fix_newlines_in_strings(raw)

try:
    data = json.loads(raw)
    print(f"✅ Loaded successfully: {len(data)} records")
    print(f"Sample: {data[0].get('Full Text', '❌ GONE')[:200]}")
    
    with open("data/consumer_complaints.json", "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)
    print("✅ Fixed JSON saved!")
    
except json.JSONDecodeError as e:
    print(f"❌ Still failing at: {e}")
    lines = raw.split('\n')
    print('\n'.join(lines[max(0, e.lineno-3):e.lineno+3]))

✅ Loaded successfully: 4318 records
Sample: Complaint Subject Product/Subject I'm complaining about: Earnings and withdrawal system on yr67.in Complaint Details: I registered on the website yr67.in with the user ID a779b6 and participated in th
✅ Fixed JSON saved!


In [28]:
import json
import os

with open("data/consumer_complaints.json", "r", encoding="utf-8") as f:
    data = json.load(f)

os.makedirs("data/txt", exist_ok=True)
saved = 0
failed = 0

for item in data:
    struct = item.get("StructuredData", {})
    unique_id = struct.get("Unique ID", "")
    full_text = item.get("Full Text", "").strip()
    title = item.get("Title/Headline", "")

    if not unique_id:
        continue

    txt_filename = f"{unique_id}.txt"

    if full_text and full_text != title:
        content = full_text
        saved += 1
    else:
        content = f"[EXTRACTION FAILED — please visit the URL manually]\n\n{title}"
        failed += 1

    with open(f"data/txt/{txt_filename}", "w", encoding="utf-8") as f:
        f.write(content)

print(f"✅ Saved: {saved} txt files with real text")
print(f"⚠️  Failed: {failed} txt files with only title")
print(f"📁 Total: {saved + failed} txt files in data/txt/")

✅ Saved: 4318 txt files with real text
⚠️  Failed: 0 txt files with only title
📁 Total: 4318 txt files in data/txt/


In [ ]:
# === RETROACTIVE DATE FIX SCRIPT ===
# Run this standalone cell to visit all records with "Unknown Date" (or missing dates)
# and accurately extract it from the individual listing's div.author-box__date element as shown in your images.
# It will periodically save progress dynamically.

import os
import json
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import random

CSV_FILE = "data/consumer_complaints.csv"
JSON_FILE = "data/consumer_complaints.json"
EXCEL_FILE = "data/consumer_complaints.xlsx"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8"
}

def fetch_date_from_url(url):
    """
    Given a URL, fetches the page and extracts the correct date.
    Implements the logic for generic links and anchor (#get-cl...) fragments.
    """
    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        if response.status_code != 200:
            return None
            
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # 1. If it's a deep-link (anchor fragment), isolate to that specific block
        if "#" in url:
            complaint_id = url.split("#")[-1].replace("get-", "")
            block = soup.find(id=complaint_id) or soup.find(id=f"get-{complaint_id}") or soup.find(attrs={"data-id": complaint_id})
            
            if block:
                # Find date specific to this complaint block
                date_tag = block.find(class_='author-box__date')
                if date_tag:
                    return date_tag.get_text(strip=True)
                    
        # 2. General logic (Matches the exact logic from the provided images)
        date_selectors = [
            ('div', {'class': 'author-box__date'}),   # Matches Image 1 and Image 2
            ('span', {'class': 'author-box__date'}),
            ('div', {'class': 'author-box__row-info'}),
            ('div', {'class': 'complaint-date'}),
            ('span', {'class': 'date'})
        ]
        
        for tag, attrs in date_selectors:
            el = soup.find(tag, attrs)
            if el:
                date_text = el.get_text(strip=True)
                # Ensure we don't accidentally grab "This thread was updated on..." if classes get meshed
                if "This thread" in date_text:
                    date_text = date_text.split("This")[0].strip()
                return date_text
                
        return None
    except Exception as e:
        print(f"Error fetching date for {url}: {e}")
        return None

print("Loading dataset to update dates...")
try:
    if not os.path.exists(JSON_FILE):
        print("JSON file not found. Please ensure data/consumer_complaints.json exists.")
    else:
        with open(JSON_FILE, "r", encoding="utf-8") as f:
            data = json.load(f)
            
        records_to_update = [item for item in data if "Unknown" in str(item.get("Original Date", "")) or not item.get("Original Date")]
        print(f"Found {len(records_to_update)} records with 'Unknown Date' or missing dates.")
        
        if len(records_to_update) > 0:
            updated_count = 0
            
            # Process records dynamically
            for idx, item in enumerate(data):
                current_date = item.get("Original Date", "")
                struct = item.get("StructuredData", {})
                
                # Check for "Unknown Date"
                if "Unknown" in str(current_date) or not current_date:
                    url = item.get("URL") or struct.get("URL")
                    if not url:
                        continue
                        
                    print(f"[{idx+1}/{len(data)}] Fixing date for: {url}")
                    time.sleep(random.uniform(2.0, 4.0)) # Polite limits so IP doesn't get blocked
                    
                    new_date = fetch_date_from_url(url)
                    if new_date:
                        # Apply cleaned date to both JSON paths
                        item["Original Date"] = new_date
                        struct["Original Date"] = new_date
                        item["StructuredData"] = struct
                        updated_count += 1
                        print(f"  -> Extracted Date: {new_date}")
                    else:
                        print("  -> Date extraction failed or block missing in DOM.")
                        
                    # Periodically save every 10 fixes to preserve progress locally
                    if updated_count > 0 and updated_count % 10 == 0:
                        with open(JSON_FILE, "w", encoding="utf-8") as f_out:
                            json.dump(data, f_out, indent=4, ensure_ascii=False)
                        print(f"--- Checkpoint Saved ({updated_count} dates fixed) ---")
                        
            # Final JSON save after complete iteration
            print("Saving final updated JSON...")
            with open(JSON_FILE, "w", encoding="utf-8") as f_out:
                json.dump(data, f_out, indent=4, ensure_ascii=False)
                
            # Reconstruct CSV & Excel so the dates populate there too!
            print("Updating CSV and Excel files...")
            if os.path.exists(CSV_FILE):
                df = pd.read_csv(CSV_FILE)
                # Map fresh dates from JSON onto DataFrame using URL
                url_to_date = {itm.get("URL"): itm.get("Original Date") for itm in data if itm.get("URL")}
                df["Original Date"] = df["URL"].map(url_to_date).fillna(df["Original Date"])
                
                df.to_csv(CSV_FILE, index=False, encoding="utf-8-sig")
                df.to_excel(EXCEL_FILE, index=False)
                print("CSV and Excel successfully updated.")
                
            print(f"\nAll completed! Fixed {updated_count} missing dates.")
        else:
            print("No missing dates found to update.")

except PermissionError:
    print("\nPermission Denied! Please close Excel if you have the CSV open and run this cell again.")
except Exception as e:
    print(f"\nAn error occurred: {e}")